In [ ]:
# TODO: move to import
def Exp(x, y, s, n=10):
    # 1 + x + x**2/2 + x**3/(3*2) - ...
    terms = [1]
    for i in range(1, n):
        terms.append(x**(i)/math.factorial(i))

    s.add(Abs(y - sum(terms)) < 10**(-n))

# Ohm's Law
Ohm's Law states that the voltage of a circuit is equal to the current times the resistance.  
Let's see how to encode that in Z3:

In [ ]:
def ohms(V, I, R, solver):
    solver.add(V == I*R)

Given a source voltage of 20 V and a total resistance of 5 $\Omega$, what is the current?

In [ ]:
s = Solver()

V, I, R = Reals('V I R')

s.add(V == 20)
s.add(R == 5)

ohms(V, I, R, s)

print(s.check())
print(s.model())
print(f"{s.model()[I]} amps (A)")

# Series Circuit

For resistors connected in series, the current flows through each resistor sequentially.
As such, the current is the same for each resistor.  
Kirchoff's Loop Law however, states that the source voltage and the total potential drop in a loop must sum to zero.  
Each potential drop across a resistor is equal to $V = IR$, which results in the following:
$$
V_{src} - V_1 - V_2 - V_3 ... = 0 \\
V_{src} = V_1 + V_2 + V_3 + ... \\
V_{src} = I R_1 + I R_2 + I R_3 + ... \\
I = \frac{V}{R_1 + R_2 + R_3 + ...} \\
I = \frac{V}{R_{eq}}
R_{eq} = R_1 + R_2 + R_3 + ... \\
$$

Therefore resistors connected in series can be replaced with one resistor equal to their sum.

We can use Z3 to find unknowns in a series circuit.
First let's add the constraints from above.

In [ ]:
def series_circuit(Vsrc, I, Req, V, R, solver):
    solver.add(Vsrc == I*Req)

    for Vi, Ri in zip(V, R):
        solver.add(Vi == I*Ri)

    solver.add(Vsrc - sum(V) == 0)

Now, consider the problem below:

A circuit has a source voltage of 10 V, and three resistors connected in series, one 10 $\Omega$, one 20 $\Omega$, and one 30 $\Omega$.  
What is the current? And what is the equivalent resistance?

In [ ]:
s = Solver()

I = Real('I')
Vsrc = Real('Vsrc')
Req = Real('Req')
R = [Real(f'R{i}') for i in range(3)]
V = [Real(f'V{i}') for i in range(3)]

s.add(R[0] == 10)
s.add(R[1] == 20)
s.add(R[2] == 30)
s.add(Vsrc == 10)

series_circuit(Vsrc, I, Req, V, R, s)

s.check()
s.model()
print(f"current: {s.model()[I]} amps")
print(f"equivalent resistance: {s.model()[Req]} ohms")

# Parallel Circuits
For resistors connected in parallel, the current flows through each resistor simultaneously.
As such, the current is split evenly across each resistor.  
Kirchoff's Junction Law however, states that the current flowing in to a junction must be equal to the current flowing out a junction.  
Each potential drop across a resistor is equal to $V = IR$, which results in the following:
$$
I = I_1 + I_2 + I_3 + ... \\
I = \frac{V_1}{R_1} + \frac{V_2}{R_2} + \frac{V_3}{R_3} + ... \\
I = V \left( \frac{1}{R_1} + \frac{1}{R_2} + \frac{1}{R_3} + ... \right) \\
I = \frac{V}{R_{eq}} \\
R_{eq} = \left( \frac{1}{R_1} + \frac{1}{R_2} + \frac{1}{R_3} + ... \right)^{-1} \\
$$

Therefore resistors connected in parallel can be replaced with a resistor equal to $R_{eq}$ shown above.

We can use Z3 to find unknowns in a parallel circuit as well.  
Let's start with the constraints from above again.

In [ ]:
def parallel_circuit(Vsrc, Itot, Req, I, R, solver):
    solver.add(Vsrc == Itot*Req)

    for Ii, Ri in zip(I, R):
        solver.add(Vsrc == Ii*Ri)

    solver.add(Itot == sum(I))

Consider the problem from before, except the resistors are now all connected in parallel:

A circuit has a source voltage of 10 V, and three resistors connected in parallel, one 10 $\Omega$, one 20 $\Omega$, and one 30 $\Omega$.  
What is the total current? And what is the equivalent resistance?

In [ ]:
s = Solver()

Vsrc = Real('Vsrc')
Itot = Real('Itot')
Req = Real('Req')
I = [Real(f'I{i}') for i in range(3)]
R = [Real(f'R{i}') for i in range(3)]

s.add(R[0] == 10)
s.add(R[1] == 20)
s.add(R[2] == 30)
s.add(Vsrc == 10)

parallel_circuit(Vsrc, Itot, Req, I, R, s)

s.check()
s.model()
print(f"current: {s.model()[Itot]} amps")
print(f"equivalent resistance: {s.model()[Req]} ohms")

# Capacitance
A capacitor stores electric charge.
Capacitance is equal to $C = Q/V$, where $Q$ is the maximum charge the capacitor can hold with applied voltage $V$.
The unit for capacitance is farads $F = \frac{C}{V}$.

Given an applied voltage of 10 V and a maximum charge of 5 mC, what is the capacitance?

In [ ]:
s = Solver()

C, Q, V = Reals('C Q V')

s.add(C == Q/V)

s.add(V == 10)
s.add(Q == 5 * 10**(-3))

print(s.check())
print(s.model())
print(f"{s.model()[C]} farads (F)")

# RC Circuits
An RC circuit is a circuit made up of resistors and capacitors.
Frequently, an RC circuit will have a "switch" in order to charge and discharge the capacitor.

When a capacitor is charging, we can use the following equation to find the charge on the capacitor:
$$ q(t) = C V_0 (1 - e^{-\frac{t}{RC}}) $$
We define the time constant $\tau = RC$, and since $C = Q/V$, we can substitute to get the following equation:
$$ q(t) = Q (1 - e^{-\frac{t}{\tau}}) $$

We can find the current by taking the derivative of the charge with respect to time:
$$
I(t) = \frac{dq}{dt} \\
I(t) = \frac{d}{dt} \left[ C V_0 (1 - e^{-\frac{t}{RC}}) \right] \\
I(t) = \frac{C V_0}{RC} e^{-\frac{t}{RC}} \\
I(t) = I_0 e^{-\frac{t}{RC}} \\
I(t) = I_0 e^{-\frac{t}{\tau}}
$$

When a capacitor is discharging, we can use the following equation to find the charge on the capacitor:
$$
q(t) = Q (e^{-\frac{t}{\tau}})
$$
Where Q is the initial charge on the capacator.

Again taking the derivative with respect to time to find the current:
$$
I(t) = \frac{dq}{dt} \\
I(t) = \frac{d}{dt} \left[ Q (e^{-\frac{t}{RC}}) \right] \\
I(t) = -\frac{Q}{RC} e^{-\frac{t}{RC}} \\
I(t) = -\frac{Q}{\tau} e^{-\frac{t}{\tau}} \\
$$

Let's encode these constraints in Z3.

In [ ]:
def rc_circuit(s, t, R, C, V0, Q, q, Vc, I0, I, Vr, tau, charging):
    exp_term = Real('e^{-t/tau}')
    Exp(-t/tau, exp_term, s)

    s.add(V0 == I0*R) # ohms law
    s.add(tau == R*C) # time constant

    # charging
    if (charging):
        s.add(Q == V0*C)  # capacitance
        s.add(q == Q*(1 - exp_term)) # charge
        s.add(I == I0*exp_term) # current

    # discharging
    else:
        s.add(q == Q*exp_term) # charge
        s.add(I == (-Q/tau)*exp_term) # current

    s.add(Vc == q/C) # voltage across capacitor
    s.add(Vr == I/R) # voltage across resistor

Consider the following problem:

There are two resistors and a capacitor connected in parallel.  
The resistors have a resistance of 100 $\Omega$ and 50 $\Omega$.  
The capacitor has a capacitance of 50 mF.  
The source voltage is 150 V.  
When the voltage across the capacitor reaches 120 V, someone flips a switch to discharge the capacitor.  
At what time does the capactior start discharging?

We can use the series constraints from before to find the equivalent resistance.

In [ ]:
s = Solver()

I = Real('I')
Vsrc = Real('Vsrc')
Req = Real('Req')
R1, R2 = Reals('R1 R2')
V1, V2 = Reals('V1 V2')

s.add(R1 == 100)
s.add(R2 == 50)
s.add(Vsrc == 150)

series_circuit(Vsrc, I, Req, [V1, V2], [R1, R2], s)

s.check()
s.model()
print(f"equivalent resistance: {s.model()[Req]} ohms")

Now we can ask Z3 to find the value of $t$ when $V_C = 120$.

In [ ]:
s = Solver()

t, R, C, V0 = Reals('t R C V_0')
Q, q, Vc = Reals('Q q V_C')
I0, I, Vr = Reals('I_0 I V_R')
tau = Real('tau')

rc_circuit(s, t, R, C, V0, Q, q, Vc, I, I0, Vr, tau, charging=True)

s.add(R == 150) # REPLACE THIS LINE
s.add(C == 50 * 10**(-3))
s.add(V0 == 150)

s.add(Vc == 120)

print(s.check()) 
print(s.model()) 
print(s.model()[t])

print(f"time: {s.model()[t].py_value():.2f} seconds") # should be ~12.07 seconds